# USPTO 短语匹配 — 微调训练（Kaggle kernel 入口，单文件）

本 notebook 即推送到 Kaggle 的运行产物（Kaggle 只执行一个 notebook）。
内部按「模块」分区，自上而下顺序执行即可：
1. **模块 1 环境配置** — 数据/模型开关、路径、设备探测
2. **模块 2 文件服务** — 依据配置实现数据读取 / 模型存取
3. **模块 3 训练入口** — 确认配置并载入数据（在此之上接微调骨架）
4. **模块 4/5 可视化** — EDA 图表生成并保存到 `outputs/figures`
5. **模块 6 模型选择** — `MODEL_KEY` 切换 deberta-v3-small / base
6. **模块 7 数据分割** — 固定种子 8:1:1 切成 train / val / test
7. **模块 8 模型定义** — `PhraseMatcher`（DeBERTa 编码器 + 回归头）、`predict_scores`、`pearson`
8. **模块 9 原生基线** — 未微调模型在 test_df 上的「地板」参考
9. **模块 10 微调训练** — Linear Probe 对照 + 全量微调 + 早停 + 测试集前后对比
10. **模块 11 推理与提交** — 对竞赛 `test.csv` 逐行预测，产出 `submission.csv`（列 `id,score`）

**开关 `USE_KAGGLE`**（见模块 1）：`True`=Kaggle 云端读写；`False`=本地读写；
`None`=自动探测 `/kaggle/input`。设备 `DEVICE` 自动检测 cuda/mps/cpu。

## 模块 1：环境配置（数据/模型开关 + 路径 + 设备探测）

In [10]:
# ============================================================
# 模块 1：环境配置（数据/模型开关 + 路径 + 设备探测）
# 翻开关：把下面 USE_KAGGLE 改成 True / False 即可强制切换。
# ============================================================
import os
from pathlib import Path

# 手动开关：None=自动探测；True=Kaggle 云端；False=本地
USE_KAGGLE = None
COMPETITION_INPUT_NAME = 'competitions/us-patent-phrase-to-phrase-matching'


def _resolve_use_kaggle() -> bool:
    env = os.environ.get('USE_KAGGLE', '').strip().lower()
    if env in ('1', 'true', 'yes', 'on'):
        return True
    if env in ('0', 'false', 'no', 'off'):
        return False
    if USE_KAGGLE is not None:
        return bool(USE_KAGGLE)
    return os.path.isdir('/kaggle/input')


ON_KAGGLE = _resolve_use_kaggle()


def _detect_device() -> str:
    '''实时探测可用计算设备：cuda > mps > cpu。
    torch 未安装时安全回退为 cpu（仅返回字符串，不依赖 torch）。'''
    try:
        import torch
        if torch.cuda.is_available():
            return 'cuda'
        mps = getattr(torch.backends, 'mps', None)
        if mps is not None and mps.is_available():
            return 'mps'
    except ImportError:
        pass
    return 'cpu'


# 当前计算设备：'cuda' / 'mps' / 'cpu'
DEVICE = _detect_device()


def get_device():
    '''返回 torch.device（需已安装 torch）；否则回退为字符串 DEVICE。
    训练代码可直接：model.to(get_device())。'''
    try:
        import torch
        return torch.device(DEVICE)
    except ImportError:
        return DEVICE


def _find_local_data() -> Path:
    '''本地模式下，向上查找含 train.csv 的 data 目录。'''
    base = Path(os.getcwd())
    for cand in [base, *base.parents]:
        if (cand / 'data' / 'train.csv').exists():
            return cand / 'data'
    return base / 'data'


LOCAL_DATA_DIR = _find_local_data()
LOCAL_MODEL_DIR = LOCAL_DATA_DIR.parent / 'train' / 'outputs'

KAGGLE_DATA_DIR = f'/kaggle/input/{COMPETITION_INPUT_NAME}'
KAGGLE_MODEL_DIR = '/kaggle/working'

DATA_DIR = KAGGLE_DATA_DIR if ON_KAGGLE else str(LOCAL_DATA_DIR)
MODEL_DIR = KAGGLE_MODEL_DIR if ON_KAGGLE else str(LOCAL_MODEL_DIR)


def summary() -> dict:
    '''返回当前开关、设备与路径概览。'''
    train_csv = (
        os.path.join(DATA_DIR, 'train.csv')
        if ON_KAGGLE
        else str(LOCAL_DATA_DIR / 'train.csv')
    )
    return {
        'ON_KAGGLE': ON_KAGGLE,
        'DEVICE': DEVICE,
        'DATA_DIR': DATA_DIR,
        'MODEL_DIR': MODEL_DIR,
        'train_csv': train_csv,
    }

## 模块 2：文件服务（数据读取 + 模型存取）

依赖模块 1 已定义的 `ON_KAGGLE` / `DATA_DIR` / `MODEL_DIR`。

In [11]:
# ============================================================
# 模块 2：文件服务（数据读取 + 模型存取）
# 依赖上方模块 1 已定义的 ON_KAGGLE / DATA_DIR / MODEL_DIR。
# ============================================================
import os


def data_path(filename: str) -> str:
    '''解析数据文件绝对路径。
    Kaggle 模式下：依次尝试 DATA_DIR、/kaggle/input 根目录，并递归搜索
    /kaggle/input 下任意子目录，找到即返回（不依赖固定目录名）。
    本地模式下：仅在 DATA_DIR 中查找。
    '''
    if ON_KAGGLE:
        candidates = [
            os.path.join(DATA_DIR, filename),
            os.path.join('/kaggle/input', filename),
        ]
        if os.path.isdir('/kaggle/input'):
            for root, _dirs, files in os.walk('/kaggle/input'):
                if filename in files:
                    candidates.append(os.path.join(root, filename))
        for c in candidates:
            if os.path.exists(c):
                return c
        top = os.listdir('/kaggle/input') if os.path.isdir('/kaggle/input') else '(目录不存在)'
        raise FileNotFoundError(
            f'Kaggle 上未找到 {filename}。\n'
            f'/kaggle/input 顶层内容: {top}\n'
            '请确认已通过「+ Add data」把比赛数据集附加到 kernel。'
        )
    cand = os.path.join(DATA_DIR, filename)
    if not os.path.exists(cand):
        raise FileNotFoundError(f'本地数据缺失: {cand}')
    return cand


def model_path(filename: str) -> str:
    '''解析模型保存/加载路径，并确保目录已创建。'''
    os.makedirs(MODEL_DIR, exist_ok=True)
    return os.path.join(MODEL_DIR, filename)


def load_train():
    '''读取训练集。'''
    import pandas as pd
    return pd.read_csv(data_path('train.csv'))


def load_test():
    '''读取测试集。'''
    import pandas as pd
    return pd.read_csv(data_path('test.csv'))


def save_model(model, filename: str) -> str:
    '''保存模型到当前 MODEL_DIR（本地或 Kaggle 云端）。'''
    path = model_path(filename)
    try:
        import torch
        torch.save(model, path)
    except ImportError:
        import pickle
        with open(path, 'wb') as f:
            pickle.dump(model, f)
    return path


def load_model(filename: str):
    '''从当前 MODEL_DIR 加载模型，与 save_model 对称。
    save_model 用 torch.save(model, path) 存的是整个模型对象，因此必须 weights_only=False；
    PyTorch>=2.6 该参数默认为 True，否则会抛 UnpicklingError。'''
    path = model_path(filename)
    try:
        import torch
        try:
            return torch.load(path, map_location='cpu', weights_only=False)
        except TypeError:      # 老版本没有 weights_only 参数
            return torch.load(path, map_location='cpu')
    except ImportError:
        import pickle
        with open(path, 'rb') as f:
            return pickle.load(f)

## 模块 1.5：运行环境自检（建议先跑这个）

在 Kaggle 上跑之前先执行本节，一眼确认：**GPU 是否可用**、**依赖是否齐全**、**数据文件是否挂载成功**。
若依赖缺失且已开启 Internet，会自动安装。

In [ ]:
# ============================================================
# 模块 1.5：运行环境自检（依赖 / GPU / 数据挂载）
# ============================================================
import importlib
import subprocess
import sys

# ---- 1) 关键依赖（缺失且在 Kaggle 上则自动安装）----
REQUIRED = [('torch', 'torch'), ('transformers', 'transformers'),
            ('numpy', 'numpy'), ('pandas', 'pandas'),
            ('sklearn', 'scikit-learn'), ('matplotlib', 'matplotlib'),
            ('seaborn', 'seaborn'), ('sentencepiece', 'sentencepiece')]

missing = []
print('=' * 62)
print('[自检] 依赖')
print('=' * 62)
for mod, pkg in REQUIRED:
    try:
        m = importlib.import_module(mod)
        print(f'  OK  {mod:<14} {getattr(m, "__version__", "?")}')
    except Exception:
        print(f'  --  {mod:<14} 缺失')
        missing.append(pkg)

if missing:
    if ON_KAGGLE:
        print(f'[自检] 自动安装缺失依赖: {missing} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + missing, check=False)
        print('[自检] 安装完成（若仍报错请手动 pip install）')
    else:
        print(f'[自检] 本地模式不自动安装，请手动: pip install {" ".join(missing)}')

# ---- 2) 计算设备 ----
print('-' * 62)
try:
    import torch
    print(f'[自检] torch {torch.__version__} | CUDA 编译版本 {torch.version.cuda}')
    print(f'[自检] CUDA 可用: {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print(f'[自检] GPU: {p.name} | 显存 {p.total_memory / 1024 ** 3:.1f} GB | '
              f'CUDA device {torch.cuda.current_device()}')
        print(f'[自检] 当前 DEVICE 解析结果: {DEVICE}')
        if DEVICE != 'cuda':
            print('[自检] 警告：DEVICE 不是 cuda，训练会跑在 CPU 上，请检查环境探测。')
    else:
        print('[自检] 警告：没有可用 GPU，训练会非常慢（Kaggle 右侧需把 Accelerator 设为 GPU）。')
except Exception as e:
    print(f'[自检] torch 检查失败: {type(e).__name__}: {e}')

# ---- 3) 数据文件挂载 ----
print('-' * 62)
print(f'[自检] ON_KAGGLE={ON_KAGGLE}')
print(f'[自检] DATA_DIR = {DATA_DIR}')
print(f'[自检] MODEL_DIR = {MODEL_DIR}')
for fn in ('train.csv', 'test.csv', 'sample_submission.csv'):
    try:
        p = data_path(fn)
        import pandas as pd
        n = len(pd.read_csv(p, nrows=1000))
        print(f'  OK  {fn:<22} {p}')
    except Exception as e:
        print(f'  --  {fn:<22} {type(e).__name__}: {str(e)[:90]}')

print('=' * 62)


## 模块 3：训练入口（示例 / 微调骨架入口）

In [12]:
# ============================================================
# 模块 3：训练入口
# 确认当前开关 / 设备 / 路径，并载入训练集（在此之上接微调骨架）。
# ============================================================
print(summary())
print('get_device() ->', get_device())

df = load_train()
print('train shape:', df.shape)
df.head()

{'ON_KAGGLE': False, 'DEVICE': 'mps', 'DATA_DIR': '/Volumes/zhitai2/git/python/USPTO/data', 'MODEL_DIR': '/Volumes/zhitai2/git/python/USPTO/train/outputs', 'train_csv': '/Volumes/zhitai2/git/python/USPTO/data/train.csv'}
get_device() -> mps
train shape: (36473, 5)


,id,anchor,target,context,score
0,37d61fd2272659b1,abatement,abatement of pollution,A47,0.50
1,7b9652b17b68b7a4,abatement,act of abating,A47,0.75
2,36d72442aefd8232,abatement,active catalyst,A47,0.25
3,5296b0c19e1ce60e,abatement,eliminating process,A47,0.50
4,54c1e3b9184cb5b6,abatement,forest region,A47,0.00


## 模块 4：数据可视化（EDA，基于 seaborn）

把 `explore.py` 的可视化能力并入单文件 notebook（本单元格定义绘图函数）。
依赖模块 1/2 的 `load_train()` 与模块 3 已载入的 `df`。
图内标签使用英文，避免 matplotlib 默认字体缺中文导致乱码；backend 设为 Agg，
无图形界面也能把图存为文件（Kaggle 上落在 /kaggle/working，可随 kernel 产出下载）。

In [13]:
# ============================================================
# 模块 4：数据可视化（基于 seaborn）
# 把 explore.py 的绘图与分析辅助函数并入单文件 notebook。
# ============================================================
import os
import pandas as pd
import matplotlib

if os.environ.get('MPLBACKEND') is None:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')


def _non_empty(d):
    if d is None or len(d) == 0:
        raise ValueError('数据为空，无法分析/绘图。')


def score_distribution(d):
    vc = d['score'].value_counts(dropna=False).sort_index()
    out = pd.DataFrame({'score': vc.index, 'count': vc.values})
    out['proportion'] = (out['count'] / len(d)).round(4)
    return out


def context_distribution(d, top_n=20):
    vc = d['context'].value_counts(dropna=False)
    out = pd.DataFrame({'context': vc.index, 'count': vc.values})
    out['proportion'] = (out['count'] / len(d)).round(4)
    return out.head(top_n) if top_n else out


def score_by_context(d):
    g = d.groupby('context')['score']
    out = pd.DataFrame({
        'n': g.size(),
        'mean_score': g.mean().round(4),
        'median_score': g.median(),
        'min_score': g.min(),
        'max_score': g.max(),
    }).sort_values('n', ascending=False)
    return out


def _tokenize(text):
    return set(str(text).lower().split())


def _jaccard(a, t):
    a, t = _tokenize(a), _tokenize(t)
    union = len(a | t)
    if union == 0:
        return 0.0
    return len(a & t) / union


def token_overlap(d):
    jac = d.apply(lambda r: _jaccard(r['anchor'], r['target']), axis=1)
    tmp = pd.DataFrame({'score': d['score'].to_numpy(), 'jaccard': jac.to_numpy()})
    g = tmp.groupby('score')['jaccard']
    return pd.DataFrame({'mean': g.mean().round(4), 'median': g.median().round(4), 'count': g.size()})


def _finalize(fig, save_path, show):
    if save_path:
        os.makedirs(os.path.dirname(os.path.abspath(save_path)), exist_ok=True)
        fig.savefig(save_path, dpi=120, bbox_inches='tight')
    if show:
        try:
            plt.show()
        except Exception:
            pass
    plt.close(fig)
    return fig


def plot_score_distribution(d, save_path=None, show=False):
    _non_empty(d)
    dist = score_distribution(d)
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.barplot(data=dist, x='score', y='count', ax=ax, color='#4C72B0')
    for _, row in dist.iterrows():
        ax.text(float(row['score']), float(row['count']), f"{float(row['proportion']):.1%}", ha='center', va='bottom', fontsize=9)
    ax.set_title('Score Distribution (label imbalance)')
    ax.set_xlabel('score')
    ax.set_ylabel('count')
    return _finalize(fig, save_path, show)


def plot_context_distribution(d, top_n=20, save_path=None, show=False):
    _non_empty(d)
    dist = context_distribution(d, top_n)
    fig, ax = plt.subplots(figsize=(9, max(4, top_n * 0.32)))
    sns.barplot(data=dist, x='count', y='context', ax=ax, color='#55A868')
    ax.set_title(f'Top-{top_n} Context (CPC class) by sample count')
    ax.set_xlabel('count')
    ax.set_ylabel('context')
    return _finalize(fig, save_path, show)


def plot_score_by_context(d, top_n=20, save_path=None, show=False):
    _non_empty(d)
    sbc = score_by_context(d).head(top_n).sort_values('mean_score')
    fig, ax = plt.subplots(figsize=(9, max(4, top_n * 0.32)))
    sns.barplot(data=sbc, x='mean_score', y=sbc.index, ax=ax, color='#C44E52')
    ax.set_title(f'Mean score by Top-{top_n} context')
    ax.set_xlabel('mean score')
    ax.set_ylabel('context')
    return _finalize(fig, save_path, show)


def plot_length_distributions(d, save_path=None, show=False):
    _non_empty(d)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, col, color in zip(axes, ('anchor', 'target'), ('#4C72B0', '#DD8452')):
        lens = d[col].astype(str).str.len()
        sns.histplot(lens, ax=ax, bins=40, color=color, kde=False)
        ax.axvline(lens.quantile(0.95), color='red', linestyle='--', label='p95')
        ax.set_title(f'{col} length (chars)')
        ax.set_xlabel('char length')
        ax.set_ylabel('count')
        ax.legend()
    fig.tight_layout()
    return _finalize(fig, save_path, show)


def plot_token_overlap(d, save_path=None, show=False):
    _non_empty(d)
    to = token_overlap(d).reset_index()
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.barplot(data=to, x='score', y='mean', ax=ax, color='#8172B3')
    for _, row in to.iterrows():
        ax.text(float(row['score']), float(row['mean']), f"{float(row['mean']):.2f}", ha='center', va='bottom', fontsize=9)
    ax.set_title('Mean token-overlap (Jaccard) by score')
    ax.set_xlabel('score')
    ax.set_ylabel('mean Jaccard')
    return _finalize(fig, save_path, show)


def plot_all(d, out_dir='figures'):
    _non_empty(d)
    names = ['score_distribution', 'context_distribution', 'score_by_context', 'length_distributions', 'token_overlap']
    paths = []
    for name in names:
        save_path = os.path.join(out_dir, f'{name}.png')
        globals()[f'plot_{name}'](d, save_path=save_path, show=False)
        paths.append(save_path)
    return paths

## 模块 5：生成可视化（保存为图片）

复用模块 3 已载入的 `df`，把 5 张图保存到 `MODEL_DIR/figures`（Kaggle 上即 `/kaggle/working/figures`，
可随 kernel 产出下载；本地即 `train/outputs/figures`）。

In [14]:
# ============================================================
# 模块 5：生成全部可视化并保存
# ============================================================
fig_dir = os.path.join(MODEL_DIR, 'figures')
paths = plot_all(df, out_dir=fig_dir)
print('可视化已保存到:', fig_dir)
for p in paths:
    size = os.path.getsize(p) if os.path.exists(p) else 'MISSING'
    print(' -', p, size)

可视化已保存到: /Volumes/zhitai2/git/python/USPTO/train/outputs/figures
 - /Volumes/zhitai2/git/python/USPTO/train/outputs/figures/score_distribution.png 28927
 - /Volumes/zhitai2/git/python/USPTO/train/outputs/figures/context_distribution.png 40246
 - /Volumes/zhitai2/git/python/USPTO/train/outputs/figures/score_by_context.png 41372
 - /Volumes/zhitai2/git/python/USPTO/train/outputs/figures/length_distributions.png 41079
 - /Volumes/zhitai2/git/python/USPTO/train/outputs/figures/token_overlap.png 26353


## 模块 6：模型选择配置

通过 `MODEL_KEY` 在 `deberta-v3-small` / `deberta-v3-base` 之间切换，后期可做精度/速度对比。
`MODEL_HF_ID` 默认是 HuggingFace 仓库名；若本地 `MODEL_DIR/pretrained/<key>` 已存在权重
（用 `modelscope` 或 `hf-mirror` 下载），则自动改用本地目录**离线加载**，避免联网拉取。

In [15]:
# ============================================================
# 模块 6：模型选择配置
# 切换 MODEL_KEY 即可选用不同尺寸的 DeBERTa-v3（用于对比）。
# ============================================================
import os

# 本地访问 HuggingFace 慢/不通时走国内镜像；Kaggle 上直连官方
if not ON_KAGGLE:
    os.environ.setdefault('HF_ENDPOINT', 'https://hf-mirror.com')

MODEL_KEY = 'deberta-v3-small'   # 亦可改 'deberta-v3-base' 做对比
MODEL_ID_MAP = {
    'deberta-v3-small': 'microsoft/deberta-v3-small',
    'deberta-v3-base':  'microsoft/deberta-v3-base',
}

# 本地预训练权重缓存目录（用 modelscope / hf-mirror 下载到此，离线加载）
PRETRAINED_DIR = os.path.join(MODEL_DIR, 'pretrained')


def resolve_model_id(key):
    '''本地 ./pretrained/<key> 权重完整则离线加载，否则回退 HuggingFace 仓库名。
    必须同时校验「权重文件」存在：只下载了 tokenizer/config 而没下权重时，
    from_pretrained 会直接报 'does not appear to have a file named pytorch_model.bin'。'''
    local = os.path.join(PRETRAINED_DIR, key)
    has_cfg = os.path.isfile(os.path.join(local, 'config.json'))
    has_weight = any(os.path.isfile(os.path.join(local, f)) for f in
                     ('model.safetensors', 'pytorch_model.bin', 'model.bin'))
    if has_cfg and has_weight:
        print(f'[模块6] 使用本地离线权重: {local}')
        return local
    if has_cfg:
        print(f'[模块6] 警告：{local} 只有 config/tokenizer、缺少权重文件，将改用在线仓库加载。')
    return MODEL_ID_MAP[key]


MODEL_HF_ID = resolve_model_id(MODEL_KEY)
FINETUNED_FILENAME = f'{MODEL_KEY}_finetuned.pt'


def finetuned_model_exists() -> bool:
    '''检测当前模型是否已有微调产物（落在本地的 MODEL_DIR）。'''
    return os.path.exists(model_path(FINETUNED_FILENAME))


print('选用模型:', MODEL_KEY, '->', MODEL_HF_ID)
print('本地已有微调模型:', finetuned_model_exists())

选用模型: deberta-v3-small -> microsoft/deberta-v3-small
本地已有微调模型: False


## 模块 7：数据分割（训练 / 验证 / 测试 = 8:1:1）

固定随机种子以保证每次分割一致，方便 small / base、微调前/后的公平对比。

In [16]:
# ============================================================
# 模块 7：数据分割（8:1:1，固定种子保证可复现）
# ============================================================
import numpy as np


def split_train_val_test(df, val_ratio=0.1, test_ratio=0.1, seed=42):
    '''按 8:1:1 随机分割，返回 (train_df, val_df, test_df)。固定种子可复现。'''
    rng = np.random.default_rng(seed)
    n = len(df)
    idx = rng.permutation(n)
    n_val = int(round(n * val_ratio))
    n_test = int(round(n * test_ratio))
    test_idx = idx[:n_test]
    val_idx = idx[n_test:n_test + n_val]
    train_idx = idx[n_test + n_val:]
    return (df.iloc[train_idx].reset_index(drop=True),
            df.iloc[val_idx].reset_index(drop=True),
            df.iloc[test_idx].reset_index(drop=True))


train_df, val_df, test_df = split_train_val_test(df)
print('分割完成 ->',
      'train:', train_df.shape[0],
      '| val:', val_df.shape[0],
      '| test:', test_df.shape[0])

分割完成 -> train: 29179 | val: 3647 | test: 3647


## 模块 8：模型与分词器构建（DeBERTa 编码器 + 回归头）

定义 `PhraseMatcher`：DeBERTa 编码器输出 pooled 表示，接 Dropout + Linear(1) + sigmoid，
把分数约束到 [0,1]。同时提供 `predict_scores`（批量推理）与 `pearson`（评估）工具函数。
本模块只定义与构造，**不含训练循环**（训练在后续模块实现）。

In [17]:
# ============================================================
# 模块 8：模型与分词器构建（仅定义/构造，不做训练）
# ============================================================
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel


class PhraseMatcher(nn.Module):
    '''DeBERTa 编码器 + 回归头：输入拼接文本，输出 [0,1] 的相似度得分。'''
    def __init__(self, model_id, hidden_dropout=0.1):
        super().__init__()
        # torch_dtype=float32：强制 fp32，避免与 fp32 回归头出现 Half/Float 不匹配
        # dtype 参数在 transformers 4.x / 5.x 之间改名过，这里做兼容：
        #   5.x 用 dtype=，4.x 忽略未知 kwargs（config 默认即 fp32）
        try:
            self.encoder = AutoModel.from_pretrained(model_id, dtype=torch.float32)
        except TypeError:
            self.encoder = AutoModel.from_pretrained(model_id, torch_dtype=torch.float32)
        hidden = self.encoder.config.hidden_size
        self.head = nn.Sequential(nn.Dropout(hidden_dropout), nn.Linear(hidden, 1))

    def forward(self, **inputs):
        # DeBERTa 不使用 token_type_ids，只传 input_ids / attention_mask
        out = self.encoder(**inputs).last_hidden_state
        cls = out[:, 0]                 # 取首 token 作为句表示
        return torch.sigmoid(self.head(cls).squeeze(-1))


def build_texts(df, tokenizer):
    '''拼接 anchor [SEP] target [SEP] context 为模型输入文本。'''
    sep = tokenizer.sep_token or ' '
    return (df['anchor'].astype(str) + sep + df['target'].astype(str)
            + sep + df['context'].astype(str)).tolist()


def predict_scores(model, tokenizer, df, device, max_len=128, batch_size=64):
    '''对 df 批量推理，返回 numpy 数组的预测分数（[0,1]）。'''
    model.eval()
    texts = build_texts(df, tokenizer)
    preds = []
    for i in range(0, len(texts), batch_size):
        enc = tokenizer(texts[i:i + batch_size], padding=True, truncation=True,
                        max_length=max_len, return_tensors='pt')
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            s = model(**enc)
        preds.append(s.cpu().numpy())
    return np.concatenate(preds)


def pearson(y_true, y_pred):
    '''Pearson 相关系数（官方评测指标）。'''
    yt = np.asarray(y_true, float)
    yp = np.asarray(y_pred, float)
    if yt.std() == 0 or yp.std() == 0:
        return float('nan')
    return float(np.corrcoef(yt, yp)[0, 1])

## 模块 9：原生模型基线预测（未微调，供后续对比）

用**未微调**的 DeBERTa（编码器预训练权重 + 随机初始化回归头）在验证/测试集上推理，
作为微调前的"地板"参考：其 Pearson 应接近 0。把预测结果存到 `MODEL_DIR/baselines`，
微调完成后用同样的 `predict_scores` 跑一遍，即可量化微调带来的提升，并公平对比 small/base。

In [ ]:
# ============================================================
# 模块 9：原生模型基线预测（未微调，仅前向推理，不训练）
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_HF_ID)
model = PhraseMatcher(MODEL_HF_ID).float().to(get_device())

val_pred = predict_scores(model, tokenizer, val_df, get_device())
test_pred = predict_scores(model, tokenizer, test_df, get_device())

val_pe = pearson(val_df['score'], val_pred)
test_pe = pearson(test_df['score'], test_pred)
print(f'基线(未微调) Pearson -> val: {val_pe:.4f} | test: {test_pe:.4f}')

# 保存基线预测，供后续微调后对比
base_dir = os.path.join(MODEL_DIR, 'baselines')
os.makedirs(base_dir, exist_ok=True)
np.save(os.path.join(base_dir, f'{MODEL_KEY}_val_pred.npy'), val_pred)
np.save(os.path.join(base_dir, f'{MODEL_KEY}_test_pred.npy'), test_pred)
print('基线预测已保存至:', base_dir)

## 模块 9（补）：原生基线预测对比（CSV + 可视化）

用 8:1:1 分割出的**测试集**（`test_df`）做原生模型预测，把「真实 label vs 预测值」存成 CSV 便于观测，并画对比图。

In [ ]:
# ============================================================
# 模块 9（补）：测试集 真实值 vs 基线预测 —— CSV + 可视化
# （接在模块 9 之后；test_df / test_pred / base_dir 已在上方计算）
# ============================================================
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# 1) 保存测试集 真实值 vs 预测值 到 CSV
compare = pd.DataFrame({
    'id': test_df['id'].values,
    'anchor': test_df['anchor'].values,
    'target': test_df['target'].values,
    'context': test_df['context'].values,
    'true_score': test_df['score'].values,
    'pred_score': test_pred,
})
csv_path = os.path.join(base_dir, f'{MODEL_KEY}_test_compare.csv')
compare.to_csv(csv_path, index=False)
print('测试集对比 CSV 已保存:', csv_path)
print(compare.head())

# 2) 可视化：真实 vs 预测（散点 + 分布对比）
pe = pearson(compare['true_score'], compare['pred_score'])
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(compare['true_score'], compare['pred_score'], alpha=0.25, s=12)
axes[0].set_xlabel('True score')
axes[0].set_ylabel('Predicted score (native baseline)')
axes[0].set_title(f'True vs Pred (test)\nPearson={pe:.4f}')

sns.histplot(compare['true_score'], color='C0', label='true', bins=20,
             ax=axes[1], stat='density', alpha=0.5)
sns.histplot(compare['pred_score'], color='C1', label='pred', bins=20,
             ax=axes[1], stat='density', alpha=0.5)
axes[1].set_xlabel('score')
axes[1].set_ylabel('density')
axes[1].set_title('Distribution: true vs pred (test)')
axes[1].legend()

plt.tight_layout()
fig_dir = os.path.join(MODEL_DIR, 'figures')
os.makedirs(fig_dir, exist_ok=True)
fig_path = os.path.join(fig_dir, f'{MODEL_KEY}_baseline_compare.png')
fig.savefig(fig_path, dpi=120)
plt.close(fig)
print('对比图已保存:', fig_path)

## 模块 10：微调训练

含两条路径对照：**Linear Probe**（冻结编码器，只训回归头）与**全量微调**（编码器+头一起训）。
每 epoch 在验证集记 Pearson/MAE，按 val Pearson 早停并保留最优权重；最终在测试集做前后对比。

In [ ]:
# ============================================================
# 模块 10：微调训练（Linear Probe 对照 + 全量微调 + 早停）
# 复用模块 8 的 PhraseMatcher / predict_scores / pearson，
# 以及模块 6 的 MODEL_HF_ID / FINETUNED_FILENAME / save_model。
# ============================================================
import torch, time
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import get_linear_schedule_with_warmup
import pandas as pd

device = get_device()
if 'tokenizer' not in globals():
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_HF_ID)

PAD_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0

# ---- 可编辑超参 ----
CFG = dict(
    epochs_probe=4, epochs_full=4,      # 2~4 轮足够，再多容易过拟合
    # ---- 学习率：头(随机初始化)要大，编码器(预训练)要小 ----
    lr_head_probe=1e-3,                 # probe 模式只训头
    lr_head_full=1e-3,                  # full 模式下头依然用大 lr
    lr_encoder_full=2e-5,               # 编码器只能小步微调
    batch_size=32, max_len=128,
    lambda_pearson=0.05,                # 叠加 (1-Pearson) 对齐官方指标（线性 ramp 生效）
    wd_probe=0.0, wd_full=0.01,         # 权重衰减（AdamW 解耦）
    warmup_ratio=0.1,                   # 前 10% 步数线性 warmup
    freeze_embeddings=True,             # 冻结词嵌入：小数据防过拟合 + 省显存
    patience=2, seed=42,
    max_grad_norm=1.0,                  # 梯度裁剪，防个别 batch 的大梯度带崩权重
)

# ---- Dataset：拼接文本 + 分数标签 ----
class PhraseDataset(Dataset):
    '''文本预先批量分词；__getitem__ 只做索引，不再逐条调用 tokenizer。'''
    def __init__(self, df, tokenizer, max_len):
        self.texts = build_texts(df, tokenizer)
        self.scores = df['score'].astype(float).values.astype('float32')
        # 批量分词：HuggingFace 内部并行处理，比逐条调用快一个数量级
        enc = tokenizer(self.texts, truncation=True, max_length=max_len)
        self.input_ids = enc['input_ids']
        self.attention_mask = enc['attention_mask']
    def __len__(self):
        return len(self.input_ids)
    def __getitem__(self, i):
        return {'input_ids': self.input_ids[i],
                'attention_mask': self.attention_mask[i],
                'label': float(self.scores[i])}


def make_collate(pad_id):
    '''按 batch 内最长序列动态 padding（比统一 pad 到 128 省近一半算力）。'''
    def collate(batch):
        n = len(batch)
        maxlen = max(len(b['input_ids']) for b in batch)
        ids = torch.full((n, maxlen), pad_id, dtype=torch.long)
        att = torch.zeros((n, maxlen), dtype=torch.long)
        for i, b in enumerate(batch):
            L = len(b['input_ids'])
            ids[i, :L] = torch.tensor(b['input_ids'], dtype=torch.long)
            att[i, :L] = torch.tensor(b['attention_mask'], dtype=torch.long)
        labels = torch.tensor([b['label'] for b in batch], dtype=torch.float32)
        return {'input_ids': ids, 'attention_mask': att, 'label': labels}
    return collate

def pearson_torch(x, y, eps=1e-6):
    '''可微 Pearson。
    eps 必须放在 sqrt 内部：若写成 denom + eps，当预测近似常数时 (x*x).sum()==0，
    sqrt 在 0 处的导数为无穷 -> 反向传播产生 NaN 梯度 -> 权重被污染。
    '''
    x = x - x.mean(); y = y - y.mean()
    denom = torch.sqrt((x * x).sum() * (y * y).sum() + eps)
    return (x * y).sum() / denom

def train_model(model, tokenizer, train_df, val_df, device, mode='full',
                epochs=4, lr_encoder=2e-5, lr_head=1e-3, batch_size=32, max_len=128,
                lambda_pearson=0.0, weight_decay=0.01, patience=3, seed=42,
                warmup_ratio=0.1, freeze_embeddings=True, max_grad_norm=1.0, pad_id=0):
    t_start = time.time()
    torch.manual_seed(seed); np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    # ---------- 步骤1：决定「谁参与训练」——靠 requires_grad ----------
    # probe：编码器全冻（False），只训头
    # full ：编码器全开（True），头照训 —— 一次 backward 两者同时更新
    for p in model.encoder.parameters():
        p.requires_grad = (mode == 'full')
    for p in model.head.parameters():
        p.requires_grad = True
    # 可选：再冻结词嵌入层（小数据下防过拟合、显著省显存）
    if freeze_embeddings and mode == 'full':
        for p in model.encoder.embeddings.parameters():
            p.requires_grad = False

    # ---------- 步骤2：分参数组设学习率（微调最关键的一步）----------
    enc_params = [p for p in model.encoder.parameters() if p.requires_grad]
    head_params = [p for p in model.head.parameters() if p.requires_grad]
    groups = []
    if enc_params:
        groups.append({'params': enc_params, 'lr': lr_encoder, 'weight_decay': weight_decay})
    groups.append({'params': head_params, 'lr': lr_head, 'weight_decay': 0.0})
    n_enc = sum(p.numel() for p in enc_params)
    n_head = sum(p.numel() for p in head_params)
    print(f'[{mode}] 可训练参数：编码器 {n_enc/1e6:.2f}M | 回归头 {n_head/1e3:.2f}K '
          f'| 组数 {len(groups)}')

    dl = DataLoader(PhraseDataset(train_df, tokenizer, max_len),
                    batch_size=batch_size, shuffle=True,
                    collate_fn=make_collate(pad_id), pin_memory=True)
    # 每个 param group 用自己的 lr（外层 lr 只是占位）
    opt = torch.optim.AdamW(groups, lr=lr_encoder, weight_decay=weight_decay)
    # 线性 warmup + 线性衰减（BERT 官方微调配方）
    from transformers import get_linear_schedule_with_warmup
    total_steps = max(1, len(dl) * epochs)
    sch = get_linear_schedule_with_warmup(
        opt, num_warmup_steps=int(total_steps * warmup_ratio), num_training_steps=total_steps)
    crit = torch.nn.MSELoss()
    ramp_steps = max(1, int(total_steps * warmup_ratio))   # Pearson 项的线性 ramp 长度
    gstep = 0
    best_pe, best_state, no_improve = -1e9, None, 0
    for ep in range(1, epochs + 1):
        model.train(); run_loss = 0.0
        for batch in dl:
            batch = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}
            labels = batch.pop('label').to(device)
            opt.zero_grad()
            preds = model(**batch)
            loss = crit(preds, labels)
            # Pearson 辅助项：从 0 线性 ramp 到 lambda_pearson；预测近似常数时跳过（防数值不稳）
            if lambda_pearson > 0:
                lam = lambda_pearson * min(1.0, gstep / ramp_steps)
                if lam > 0 and preds.std() > 1e-6:
                    loss = loss + lam * (1 - pearson_torch(preds, labels))
            # 一次 backward 同时给「头」和「编码器」算梯度 → 两块一起更新
            loss.backward()
            if max_grad_norm and max_grad_norm > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            opt.step()   # 按各自 param group 的 lr 更新
            sch.step()   # 推进 warmup/decay 调度
            gstep += 1
            run_loss += loss.item() * labels.size(0)
        val_pred = predict_scores(model, tokenizer, val_df, device, max_len=max_len, batch_size=batch_size)
        val_pe = pearson(val_df['score'].values, val_pred)
        val_mae = float(np.mean(np.abs(val_df['score'].values - val_pred)))
        print(f'[{mode}] ep{ep}/{epochs} loss={run_loss/len(dl.dataset):.4f} '
              f'val_pearson={val_pe:.4f} val_mae={val_mae:.4f}')
        if val_pe > best_pe:
            best_pe, best_state, no_improve = val_pe, \
                {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'[{mode}] 早停 @ ep{ep}'); break
    if best_state is not None:
        model.load_state_dict(best_state)   # 回滚到 val 最优那版权重
    mem = torch.cuda.max_memory_allocated() / (1024 ** 3) if torch.cuda.is_available() else 0.0
    print(f'[{mode}] 最佳 val_pearson={best_pe:.4f} | 耗时 {time.time()-t_start:.1f}s '
          f'| 峰值显存 {mem:.2f} GB')
    return model, best_pe

# ---- 1) Linear Probe：冻结编码器，只训头 ----
# 改进 C：先固定随机种子再构造模型，保证 probe / full 的回归头初值完全一致（对照才公平）
torch.manual_seed(CFG['seed']); np.random.seed(CFG['seed'])
print('=== Linear Probe（冻结编码器，仅训回归头）===')
model_probe = PhraseMatcher(MODEL_HF_ID).float().to(device)
model_probe, pe_probe = train_model(
    model_probe, tokenizer, train_df, val_df, device, mode='probe',
    epochs=CFG['epochs_probe'], lr_head=CFG['lr_head_probe'],
    batch_size=CFG['batch_size'], max_len=CFG['max_len'],
    lambda_pearson=CFG['lambda_pearson'], weight_decay=CFG['wd_probe'],
    patience=CFG['patience'], seed=CFG['seed'],
    warmup_ratio=CFG['warmup_ratio'], freeze_embeddings=CFG['freeze_embeddings'],
    max_grad_norm=CFG['max_grad_norm'], pad_id=PAD_ID)

# ---- 2) 全量微调：编码器+头一起训 ----
torch.manual_seed(CFG['seed']); np.random.seed(CFG['seed'])
print('=== 全量微调（编码器 + 回归头）===')
model_full = PhraseMatcher(MODEL_HF_ID).float().to(device)
# 自检：记录训练前的词嵌入，用于验证 freeze_embeddings 是否真的生效
_emb0 = model_full.encoder.embeddings.word_embeddings.weight.detach().clone()
model_full, pe_full = train_model(
    model_full, tokenizer, train_df, val_df, device, mode='full',
    epochs=CFG['epochs_full'],
    lr_encoder=CFG['lr_encoder_full'], lr_head=CFG['lr_head_full'],
    batch_size=CFG['batch_size'], max_len=CFG['max_len'],
    lambda_pearson=CFG['lambda_pearson'], weight_decay=CFG['wd_full'],
    patience=CFG['patience'], seed=CFG['seed'],
    warmup_ratio=CFG['warmup_ratio'], freeze_embeddings=CFG['freeze_embeddings'],
    max_grad_norm=CFG['max_grad_norm'], pad_id=PAD_ID)

# 自检：词嵌入平均变化量（冻结时应为 0.00000000，解冻时应 > 0）
_emb1 = model_full.encoder.embeddings.word_embeddings.weight
_emb_delta = (_emb1 - _emb0).norm(dim=1).mean().item()
print(f'[自检] 词嵌入平均变化量 = {_emb_delta:.8f}')

# ---- 保存全量微调最佳权重 ----
save_path = save_model(model_full, FINETUNED_FILENAME)
print('微调模型已保存:', save_path, '| finetuned_model_exists():', finetuned_model_exists())
print('=== 微调效果对比 (val) ===')
print('基线(未微调) : 见模块9（≈随机，Pearson≈0）')
print(f'Linear Probe : val_pearson={pe_probe:.4f}')
print(f'全量微调     : val_pearson={pe_full:.4f}')

## 模块 10（补）：测试集 微调前后对比（CSV + 可视化）

In [ ]:
# ============================================================
# 模块 10（补）：测试集 微调前后对比（CSV + 可视化）
# ============================================================
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

device = get_device()
base_dir = os.path.join(MODEL_DIR, 'baselines')
os.makedirs(base_dir, exist_ok=True)

# 微调后预测
test_pred_tuned = predict_scores(model_full, tokenizer, test_df, device, max_len=128, batch_size=64)
# Linear Probe 预测（改进 F：把 probe 也放进测试集对比，验证 probe < full）
test_pred_probe = predict_scores(model_probe, tokenizer, test_df, device, max_len=128, batch_size=64)
# 原生基线预测（同 run 内重算，保证可比）
base_model = PhraseMatcher(MODEL_HF_ID).float().to(device)
test_pred_base = predict_scores(base_model, tokenizer, test_df, device, max_len=128, batch_size=64)

y_true = test_df['score'].values
pe_base = pearson(y_true, test_pred_base)
pe_probe = pearson(y_true, test_pred_probe)
pe_tuned = pearson(y_true, test_pred_tuned)
mae_base = float(np.mean(np.abs(y_true - test_pred_base)))
mae_probe = float(np.mean(np.abs(y_true - test_pred_probe)))
mae_tuned = float(np.mean(np.abs(y_true - test_pred_tuned)))
print('===== 测试集对比（test_df，来自 8:1:1 分割）=====')
print(f'未微调基线  : Pearson={pe_base:.4f}  MAE={mae_base:.4f}')
print(f'Linear Probe: Pearson={pe_probe:.4f}  MAE={mae_probe:.4f}   (Δ vs base = {pe_probe-pe_base:+.4f})')
print(f'全量微调    : Pearson={pe_tuned:.4f}  MAE={mae_tuned:.4f}   (Δ vs base = {pe_tuned-pe_base:+.4f}, Δ vs probe = {pe_tuned-pe_probe:+.4f})')

compare = pd.DataFrame({
    'id': test_df['id'].values,
    'anchor': test_df['anchor'].values,
    'target': test_df['target'].values,
    'context': test_df['context'].values,
    'true_score': test_df['score'].values,
    'base_pred_score': test_pred_base,
    'probe_pred_score': test_pred_probe,
    'finetune_pred_score': test_pred_tuned,
})
csv_path = os.path.join(base_dir, f'{MODEL_KEY}_test_finetune_compare.csv')
compare.to_csv(csv_path, index=False)
print('微调对比 CSV 已保存:', csv_path)
print(compare.head())

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
axes[0].scatter(compare['true_score'], compare['finetune_pred_score'], alpha=0.25, s=12)
axes[0].set_xlabel('True score')
axes[0].set_ylabel('Predicted (finetuned)')
axes[0].set_title(f'Finetuned True vs Pred (test)\nPearson={pe_tuned:.4f}')
axes[1].scatter(compare['true_score'], compare['probe_pred_score'], alpha=0.25, s=12, color='C1')
axes[1].set_xlabel('True score')
axes[1].set_ylabel('Predicted (linear probe)')
axes[1].set_title(f'Linear Probe True vs Pred (test)\nPearson={pe_probe:.4f}')
sns.histplot(compare['true_score'], color='C0', label='true', bins=20, ax=axes[2], stat='density', alpha=0.5)
sns.histplot(compare['probe_pred_score'], color='C1', label='probe', bins=20, ax=axes[2], stat='density', alpha=0.5)
sns.histplot(compare['finetune_pred_score'], color='C2', label='finetuned', bins=20, ax=axes[2], stat='density', alpha=0.5)
axes[2].set_xlabel('score'); axes[2].set_ylabel('density'); axes[2].legend()
axes[2].set_title('Distribution: true vs probe vs finetuned (test)')
plt.tight_layout()
fig_dir = os.path.join(MODEL_DIR, 'figures')
os.makedirs(fig_dir, exist_ok=True)
fig_path = os.path.join(fig_dir, f'{MODEL_KEY}_finetune_compare.png')
fig.savefig(fig_path, dpi=120); plt.close(fig)
print('微调对比图已保存:', fig_path)

## 模块 11：测试集推理与提交文件生成

用**微调后的模型**对竞赛 `test.csv` 逐行预测相似度，产出可直接提交的 `submission.csv`。

- 输入：`data/test.csv`（列 `id, anchor, target, context`，**无 score 列**）
- 输出：`submission.csv`，列固定为 **`id,score`**，行序与 `test.csv` 完全一致
  （与官方 `sample_submission.csv` 对齐，会自动校验 id 顺序）
- 另存一份**明细版** `*_test_predictions.csv`（含 anchor/target/context/pred），便于人工核查
- 模型来源优先级：内存中的 `model_full`（模块 10 刚训完）→ 磁盘 `MODEL_DIR/{MODEL_KEY}_finetuned.pt`


In [ ]:
# ============================================================
# 模块 11：测试集推理与提交文件生成
# 依赖：模块 8（PhraseMatcher / predict_scores）、模块 6（MODEL_KEY / MODEL_HF_ID /
#        FINETUNED_FILENAME / model_path）、模块 2（load_test / data_path）
# 输出：submission.csv（列 = id,score，顺序与 test.csv 一致）
# ============================================================
import os
import numpy as np
import pandas as pd
import torch

device = get_device()
SUB_BATCH_SIZE = 64

if 'tokenizer' not in globals():
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_HF_ID)


# ---------- 1) 取到可用的推理模型 ----------
def _resolve_infer_model():
    '''优先用内存中的 model_full（模块 10 刚训完）；否则从磁盘加载微调产物。'''
    for name in ('model_full', 'model_probe'):
        m = globals().get(name, None)
        if isinstance(m, torch.nn.Module):
            print(f'[模块11] 使用内存中的模型: {name}')
            return m, name
    path = model_path(FINETUNED_FILENAME)
    if os.path.exists(path):
        try:
            m = torch.load(path, map_location='cpu', weights_only=False)  # PyTorch>=2.6 需显式关闭
        except TypeError:
            m = torch.load(path, map_location='cpu')
        print(f'[模块11] 从磁盘加载微调模型: {path}')
        return m, 'disk'
    raise FileNotFoundError(
        f'[模块11] 未找到微调模型：{path}\n'
        f'请先运行模块 10（微调训练），或确认 {FINETUNED_FILENAME} 已存在。')


infer_model, model_tag = _resolve_infer_model()
infer_model = infer_model.to(device).eval()


# ---------- 2) 读取待预测的 test 文件 ----------
def load_test_frame():
    '''优先用竞赛官方 test.csv；找不到则退回本地 8:1:1 切出的 test_df（仅本地调试用）。'''
    try:
        df = load_test()
        print(f'[模块11] 已读取竞赛 test.csv：{df.shape} 列={list(df.columns)}')
        return df, 'competition_test.csv'
    except Exception as e:
        print(f'[模块11] 读取竞赛 test.csv 失败（{type(e).__name__}: {e}）')
        print('[模块11] ⚠️ 警告：已退回本地 8:1:1 切分的 test_df。')
        print('[模块11] ⚠️ 其 id 来自 train.csv，不是竞赛 test.csv 的 id，'
              '本次生成的 submission.csv 不可用于正式提交（仅供本地流程验证）。')
        return test_df, 'local_test_df(仅调试,不可提交)'


test_infer, test_source = load_test_frame()

for _col in ('anchor', 'target', 'context'):
    if _col not in test_infer.columns:
        raise KeyError(f'[模块11] test 文件缺少列 "{_col}"，无法拼接模型输入。实际列：{list(test_infer.columns)}')


# ---------- 3) 批量推理 ----------
def predict_frame(model, df, max_len=128, batch_size=SUB_BATCH_SIZE):
    '''对任意含 anchor/target/context 的 DataFrame 批量预测，返回 float 数组。'''
    pred = predict_scores(model, tokenizer, df, device, max_len=max_len, batch_size=batch_size)
    pred = np.asarray(pred, dtype=float)
    if not np.isfinite(pred).all():
        bad = int((~np.isfinite(pred)).sum())
        raise ValueError(f'[模块11] 预测出现 {bad} 个 NaN/Inf，请检查模型是否训练异常。')
    return pred


# max_len：优先用模块 10 的 CFG；单独重跑本模块时回退到 128
MAX_LEN = CFG.get('max_len', 128) if 'CFG' in globals() else 128

print('[模块11] 开始推理 ...')
test_pred = predict_frame(infer_model, test_infer, max_len=MAX_LEN)
print(f'[模块11] 推理完成：{len(test_pred)} 条 | '
      f'min={test_pred.min():.4f} max={test_pred.max():.4f} '
      f'mean={test_pred.mean():.4f} std={test_pred.std():.4f}')


# ---------- 4) 生成提交文件（列固定 id,score，顺序与 test.csv 一致）----------
def build_submission(df, pred, out_path, sample_csv=None):
    '''构造并落盘提交文件；返回 (DataFrame, 校验信息)。'''
    if 'id' not in df.columns:
        raise KeyError('[模块11] test 文件缺少 id 列，无法生成提交文件。')
    sub = pd.DataFrame({'id': df['id'].values, 'score': np.asarray(pred, dtype=float)})
    assert list(sub.columns) == ['id', 'score'], f'提交列名必须为 id,score，实际 {list(sub.columns)}'
    assert len(sub) == len(df), '提交行数与 test 行数不一致'
    assert sub['id'].duplicated().sum() == 0, '提交文件出现重复 id'

    info = []
    if sample_csv is not None and os.path.exists(sample_csv):
        sample = pd.read_csv(sample_csv)
        ok_cols = list(sample.columns) == ['id', 'score']
        ok_len = len(sample) == len(sub)
        ok_ids = bool((sample['id'].values == sub['id'].values).all())
        if ok_cols and ok_len and ok_ids:
            info.append(f'已与 {os.path.basename(sample_csv)} 对齐（列名/行数/id 顺序全部一致）')
        else:
            info.append(f'警告：与 {os.path.basename(sample_csv)} 不一致 '
                        f'(列名={ok_cols}, 行数={ok_len}, id顺序={ok_ids})')
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    sub.to_csv(out_path, index=False)
    return sub, info


# 提交文件：标准名 submission.csv（Kaggle 上即 /kaggle/working/submission.csv）
sub_path = os.path.join(MODEL_DIR, 'submission.csv')
# 归档一份带模型名的副本
sub_arch_dir = os.path.join(MODEL_DIR, 'submissions')
os.makedirs(sub_arch_dir, exist_ok=True)
sub_arch_path = os.path.join(sub_arch_dir, f'{MODEL_KEY}_submission.csv')

sample_csv = None
for cand in (os.path.join(DATA_DIR, 'sample_submission.csv'),
             os.path.join(os.path.dirname(data_path('train.csv')), 'sample_submission.csv')):
    if os.path.exists(cand):
        sample_csv = cand
        break

submission, sub_info = build_submission(test_infer, test_pred, sub_path, sample_csv)
sub_df = pd.read_csv(sub_path, header=0)   # 回读校验
assert list(sub_df.columns) == ['id', 'score'], '回读后列名异常'
assert len(sub_df) == len(test_infer), '回读后行数异常'
submission.to_csv(sub_arch_path, index=False)

for msg in sub_info:
    print('[模块11]', msg)
print('[模块11] 提交文件已保存:', sub_path)
print('[模块11] 归档副本已保存:', sub_arch_path)


# ---------- 5) 明细版（含原文，便于人工核查 / 后续特征工程）----------
detail = test_infer.copy()
detail['pred_score'] = test_pred
detail_path = os.path.join(MODEL_DIR, 'baselines', f'{MODEL_KEY}_test_predictions.csv')
os.makedirs(os.path.dirname(detail_path), exist_ok=True)
detail.to_csv(detail_path, index=False)
print('[模块11] 明细预测已保存:', detail_path)


# ---------- 6) 预览 ----------
print('=' * 60)
print('[模块11] 提交文件前 5 行:')
print(submission.head().to_string(index=False))
print('-' * 60)
print('[模块11] 明细文件前 3 行:')
print(detail.head(3).to_string(index=False))
print('=' * 60)
print(f'[模块11] 完成 | 数据源={test_source} | 模型={model_tag} | 行数={len(submission)}')
